In [ ]:
# =====================================================
# UMAP visualization + Global Pie Charts + ARI Heatmap
# - One UMAP per model, recolored by multiple tasks
# - ONE pie chart per task (not per model), with counts on slices
# - Colors are consistent between UMAPs and pies
# - Robust outlier removal in UMAP plots (MAD/IQR/percentile)
#   * Outliers are filtered ONLY for plotting; pies & ARI unaffected
#   * Saves both raw and filtered UMAP coords for transparency
# - NEW: ARI experiment (KMeans on embeddings) + heatmap
#   * For each model & task, cluster in embedding space and compute ARI
#   * Saves CSV + publication-ready heatmap
# =====================================================
import os
import re
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap

from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

# -----------------------------
# Config
# -----------------------------
FAST_MODE = True
MIN_CLASS_COUNT = 100 if FAST_MODE else 50
RANDOM_SEED = 6740

# ====== EDIT THESE: your fine-tuned FM run folders (each contains run_params.json + embeddings_all/) ======
# Choose which FM embeddings to visualize:
#   "best" -> embeddings_all/image_feats.npy
#   "last" -> embeddings_all/image_feats_last.npy
FM_EMB_VERSION = "last"   # options: "best", "last"
# =========================================================================================================

# Off-the-shelf baselines (embedding folders must contain image_feats.npy + index.csv)
BASELINE_EMB_SETS = {
    "MAE-ViT-B/16 – Pretrained":    os.path.join("pretrained_feats2", "msi_multitask_mae_untrained"),
    "DINOv2-ViT-B/14 – Pretrained":    os.path.join("pretrained_feats2", "msi_multitask_dinov2_untrained"),
    "MAE-ViT-B/16 – Fine-tuned":    os.path.join("pretrained_feats2", "msi_multitask_mae"),
    "DINOv2-ViT-B/14 – Fine-tuned":    os.path.join("pretrained_feats2", "msi_multitask_dinov2"),
}

# Metadata sources
IDX_PARQUET = "metaspace_images_dump/msi_fm_samples3.parquet"
MAN_PARQUET = "metaspace_images_dump/manifest_expanded.parquet"

# Output (single combined folder for all models)
OUT_DIR = os.path.join("fm_ssl_run", "baseline_eval_combined2", "umaps")
os.makedirs(OUT_DIR, exist_ok=True)

# UMAP params (per model)
N_NEIGHBORS = 15
MIN_DIST = 0.15
METRIC = "cosine"

# Tasks to color by (must exist after canonicalization)
TASKS = ["organism", "polarity", "Organism_Part", "Condition", "analyzerType", "ionisationSource"]

# ------------- Outlier Removal (for plotting only) -------------
OUTLIER_REMOVE = True
OUTLIER_METHOD = "mad"   # options: "mad", "iqr", "percentile"
MAD_K = 3                # higher = fewer removals
IQR_K = 2.5              # multiplier for Q3 + k*IQR on radial distances
DIST_PERCENTILE = 99.7   # keep points within this distance percentile (if method="percentile")
# Safety rails: if we accidentally drop too much, we auto-relax once
MIN_KEEP_FRACTION = 0.85

# ------------- ARI experiment config -------------
ARI_MAX_SAMPLES_PER_TASK = 80_000   # cap per task/model to keep k-means reasonable
ARI_N_INIT = 10                     # k-means restarts

sns.set_style("white")

# -----------------------------
# Helpers for FM-run naming & paths
# -----------------------------
def read_vit_name_from_run(run_dir: str) -> str:
    """Read cfg.vit_name from run_params.json; fallback to folder name."""
    rp = os.path.join(run_dir, "run_params.json")
    if os.path.exists(rp):
        try:
            with open(rp, "r", encoding="utf-8") as f:
                data = json.load(f)
            if "cfg" in data and isinstance(data["cfg"], dict):
                vit = data["cfg"].get("vit_name", None)
                if vit:
                    return str(vit)
            vit = data.get("vit_name", None)
            if vit:
                return str(vit)
        except Exception as e:
            print(f"[WARN] Failed reading vit_name from {rp}: {e}")
    return os.path.basename(os.path.normpath(run_dir))

def make_unique_name(base_name: str, existing: set, hint: str) -> str:
    """Ensure display name uniqueness across runs/baselines."""
    if base_name not in existing:
        return base_name
    short = hint.replace("\\", "/").strip("/").split("/")[-1]
    cand = f"{base_name} ({short})"
    if cand not in existing:
        return cand
    tiny = hashlib.md5(hint.encode("utf-8")).hexdigest()[:6]
    return f"{base_name} [{tiny}]"

def fm_embeddings_dir(run_dir: str) -> str:
    """Default FM embeddings location within each run."""
    return os.path.join(run_dir, "embeddings_all")

def fm_feats_filename() -> str:
    return "image_feats.npy" if FM_EMB_VERSION.lower() == "best" else "image_feats_last.npy"

def fm_embeddings_ready(emb_dir: str) -> bool:
    feats_file = fm_feats_filename()
    return (
        os.path.exists(os.path.join(emb_dir, feats_file)) and
        os.path.exists(os.path.join(emb_dir, "index.csv"))
    )

def baseline_embeddings_ready(emb_dir: str) -> bool:
    return (
        os.path.exists(os.path.join(emb_dir, "image_feats.npy")) and
        os.path.exists(os.path.join(emb_dir, "index.csv"))
    )

# -----------------------------
# Canonicalization (matches eval script)
# -----------------------------
def _clean(s):
    if pd.isna(s): return None
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

def canonicalize_labels(df):
    df = df.copy()

    # 1) Polarity
    pol_map = {"pos":"Positive","positive":"Positive","+":"Positive",
               "neg":"Negative","negative":"Negative","-":"Negative"}
    def canon_polarity(s):
        if s is None: return None
        t = _clean(s).lower()
        t2 = pol_map.get(t, t)
        if t2 in ("positive","negative"):
            return t2.capitalize()
        if "pos" in t: return "Positive"
        if "neg" in t: return "Negative"
        return _clean(s)
    if "polarity" in df.columns:
        df["polarity"] = df["polarity"].map(canon_polarity)

    # 2) Ionisation Source
    def canon_ion_src(s):
        if s is None: return None
        t_raw = _clean(s)
        t = t_raw.upper().replace("-", "").replace("_","")
        if "APSMALDI" in t: return "AP-SMALDI"
        if "IRMALDESI" in t or "IRMALDI" in t: return "IR-MALDESI"
        if "APMALDI" in t: return "AP-MALDI"
        if "DESIMSI" in t: return "DESI"
        if "DESI" in t: return "DESI"
        if "MALDI" in t: return "MALDI"
        return t_raw
    if "ionisationSource" in df.columns:
        df["ionisationSource"] = df["ionisationSource"].map(canon_ion_src)

    # 3) Analyzer Type
    def canon_analyzer(s):
        if s is None: return None
        t = _clean(s); tl = t.lower()
        if "timstof" in tl and "flex" in tl: return "timsTOF Flex"
        if "fticr" in tl:
            if "12t" in tl: return "12T FTICR"
            if "7t" in tl and "scimax" in tl: return "FTICR scimaX 7T"
            return "FTICR"
        if "orbitrap" in tl or "q-exactive" in tl: return "Orbitrap"
        if "tof" in tl and "reflector" in tl: return "TOF reflector"
        if tl.strip() == "qtof": return "qTOF"
        return t
    if "analyzerType" in df.columns:
        df["analyzerType"] = df["analyzerType"].map(canon_analyzer)

    # 4) Organism
    def canon_organism(s):
        if s is None: return None
        t = _clean(s); tl = t.lower()
        if "|" in t or "," in t:
            if ("human" in tl or "homo sapiens" in tl) and ("mouse" in tl or "mus musculus" in tl):
                return "Mixed"
        if "homo sapiens" in tl or tl.strip() in {"human","h. sapiens","homo"}:
            return "Homo sapiens"
        if "mus musculus" in tl or tl.strip() in {"mouse","m. musculus"}:
            return "Mus musculus"
        return t
    if "organism" in df.columns:
        df["organism"] = df["organism"].map(canon_organism)

    # 5) Organism_Part
    def canon_part(s):
        if s is None: return None
        t = _clean(s); tl = t.lower()
        if "kidney" in tl: return "Kidney"
        if "brain"  in tl: return "Brain"
        if "liver"  in tl: return "Liver"
        if "lung"   in tl: return "Lung"
        if "breast" in tl: return "Breast"
        if "skin"   in tl: return "Skin"
        if "heart"  in tl or "cardiac" in tl: return "Heart"
        return t
    if "Organism_Part" in df.columns:
        df["Organism_Part"] = df["Organism_Part"].map(canon_part)

    # 6) Condition
    def canon_condition(s):
        if s is None: return None
        t = _clean(s); tl = t.lower()
        if tl in {"n/a","na","none","not available",""}: return "NA"
        if tl in {"biopsy","biopsies"}: return "Biopsy"
        if "fresh frozen" in tl or "frozen" in tl: return "Frozen"
        if "tumor" in tl or "tumour" in tl: return "Tumor"
        if "cancer" in tl: return "Cancer"
        if "wildtype" in tl or tl == "wt": return "Wildtype"
        if "healthy" in tl or "control" in tl: return "Healthy"
        if "diseased" in tl or "disease" in tl: return "Diseased"
        return t
    if "Condition" in df.columns:
        df["Condition"] = df["Condition"].map(canon_condition)

    return df

# -----------------------------
# Load metadata (+ canonicalize)
# -----------------------------
idx = pd.read_parquet(IDX_PARQUET)
man = pd.read_parquet(MAN_PARQUET)
meta = idx.merge(man, on="dataset_id", how="left", suffixes=("", "_man"))
meta = meta.drop_duplicates("sample_path", keep="first").reset_index(drop=True)
meta = canonicalize_labels(meta)

# -----------------------------
# Build model list (FM runs + baselines)
# -----------------------------
MODELS = []  # {"tag": display_name, "emb_dir": path, "feats_file": fname}
seen_names = set()

# Add baselines
for tag, emb_dir in BASELINE_EMB_SETS.items():
    name_disp = make_unique_name(tag, seen_names, hint=emb_dir)
    seen_names.add(name_disp)
    if not baseline_embeddings_ready(emb_dir):
        print(
            f"[WARN] Missing embeddings for baseline '{name_disp}' at {emb_dir} "
            f"(need image_feats.npy + index.csv); skipping."
        )
        continue
    MODELS.append({"tag": name_disp, "emb_dir": emb_dir, "feats_file": "image_feats.npy"})

if not MODELS:
    raise SystemExit("[ERR] No models to visualize. Check FM_RUN_DIRS and baseline paths.]")

print("[INFO] Models to visualize:")
for m in MODELS:
    print(f" - {m['tag']}: {m['emb_dir']} [{m['feats_file']}]")

# -----------------------------
# Outlier detection helpers
# -----------------------------
def _mad(x):
    med = np.median(x)
    return np.median(np.abs(x - med))

def outlier_mask_umap(umap_xy: np.ndarray,
                      method: str = OUTLIER_METHOD,
                      mad_k: float = MAD_K,
                      iqr_k: float = IQR_K,
                      dist_pct: float = DIST_PERCENTILE) -> np.ndarray:
    """
    Return boolean mask of points to KEEP based on robust radial distance
    from the median center of the UMAP cloud.
    """
    if umap_xy.ndim != 2 or umap_xy.shape[1] != 2:
        return np.ones(len(umap_xy), dtype=bool)

    # robust center
    cx, cy = np.median(umap_xy[:, 0]), np.median(umap_xy[:, 1])
    r = np.sqrt((umap_xy[:, 0] - cx) ** 2 + (umap_xy[:, 1] - cy) ** 2)

    if method == "mad":
        mad = _mad(r)
        if mad == 0:
            # fallback to percentile if degenerate
            thr = np.percentile(r, dist_pct)
            keep = r <= thr
        else:
            thr = np.median(r) + mad_k * mad
            keep = r <= thr

    elif method == "iqr":
        q1, q3 = np.percentile(r, [25, 75])
        iqr = q3 - q1
        thr = q3 + iqr_k * iqr
        keep = r <= thr

    elif method == "percentile":
        thr = np.percentile(r, dist_pct)
        keep = r <= thr

    else:
        # unknown method -> keep all
        return np.ones(len(umap_xy), dtype=bool)

    # Safety rail: if we dropped too many, relax threshold once
    frac = keep.mean()
    if frac < MIN_KEEP_FRACTION:
        print(f"[WARN] Outlier removal kept only {frac:.1%}; relaxing...")
        thr_relaxed = np.percentile(r, min(99.95, max(dist_pct, 99.0)))
        keep = r <= thr_relaxed
    return keep

# -----------------------------
# UMAP helper (fit/transform)
# -----------------------------
def run_umap_once(X, n_neighbors=30, min_dist=0.15, seed=6740, metric="cosine"):
    reducer = umap.UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        random_state=seed,
        metric=metric
    )
    return reducer.fit_transform(X)

# -----------------------------
# 1) Build per-task kept-classes + color maps (global, used by UMAPs & pies)
# -----------------------------
def build_task_color_maps(meta_df, tasks, min_class_count=50):
    kept_by_task = {}
    cmap_by_task = {}
    for task in tasks:
        if task not in meta_df.columns:
            continue
        labels_full = meta_df[task].astype("string").fillna("NA")
        vc = labels_full.value_counts()
        kept = sorted(vc[vc >= min_class_count].index.tolist())
        palette = (
            sns.color_palette("tab10", n_colors=max(len(kept), 1))
            if len(kept) <= 10
            else sns.color_palette("husl", max(len(kept), 1))
        )
        color_map = {lab: palette[i] for i, lab in enumerate(kept)}
        kept_by_task[task] = kept
        cmap_by_task[task] = color_map
    return kept_by_task, cmap_by_task

# Build once (so all models/plots use identical task colors)
KEPT_BY_TASK, CMAP_BY_TASK = build_task_color_maps(meta, TASKS, MIN_CLASS_COUNT)

# -----------------------------
# 2) Global pies (one per task), using same colors as UMAPs
# -----------------------------
def save_global_pie_for_task(meta_df, task, kept_by_task, cmap_by_task, out_dir):
    if task not in meta_df.columns:
        print(f"[WARN] Global pie: task '{task}' not in metadata; skipping.")
        return

    labels = meta_df[task].astype("string").fillna("NA")
    vc_all = labels.value_counts()
    kept = kept_by_task.get(task, [])
    c_map = cmap_by_task.get(task, {})

    # Only plot kept classes
    sizes, colors, names = [], [], []
    total = 0
    for lab in kept:
        cnt = int(vc_all.get(lab, 0))
        if cnt > 0:
            sizes.append(cnt)
            colors.append(c_map[lab])
            names.append(str(lab))
            total += cnt

    if total == 0:
        print(f"[WARN] Global pie: '{task}' has no kept samples; skipping.")
        return

    def _count_autopct(pct):
        absolute = int(round(pct * total / 100.0))
        return f"{absolute}"

    fig, ax = plt.subplots(figsize=(6.4, 6.4))
    wedges, texts, autotexts = ax.pie(
        sizes,
        colors=colors,
        startangle=90,
        autopct=_count_autopct,
        pctdistance=0.72,
        textprops={"fontsize": 9},
        wedgeprops=dict(linewidth=0.5, edgecolor="white"),
    )

    plt.setp(autotexts, size=16, weight="bold", color="black")
    plt.setp(texts, size=11)

    ax.axis("equal")

    ax.legend(
        wedges, names,
        title="Classes",
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        frameon=False,
        fontsize=8
    )

    plt.tight_layout()
    out_path = os.path.join(out_dir, f"pie_{task}.png")
    plt.savefig(out_path, dpi=250, bbox_inches="tight", pad_inches=0.15)
    plt.close()
    print(f"[OK] saved: {out_path}")

# -----------------------------
# 3) UMAP plotting using precomputed task color maps
#     (applies outlier removal to the scatter only)
# -----------------------------
def plot_same_coords_color_by_tasks(umap_2d, df_join, model_tag, out_dir,
                                    tasks, min_class_count=50,
                                    outlier_remove=True):
    assert len(df_join) == len(umap_2d), "UMAP coords and df_join length mismatch"

    # Save RAW coords for reuse
    raw_coords_out = os.path.join(out_dir, f"umap_coords_{model_tag}_raw.csv")
    pd.DataFrame({
        "sample_path": df_join["sample_path"].values,
        "umap_x": umap_2d[:,0],
        "umap_y": umap_2d[:,1],
    }).to_csv(raw_coords_out, index=False)

    # Compute outlier mask (for plotting)
    if outlier_remove:
        keep_mask = outlier_mask_umap(umap_2d, method=OUTLIER_METHOD,
                                      mad_k=MAD_K, iqr_k=IQR_K, dist_pct=DIST_PERCENTILE)
    else:
        keep_mask = np.ones(len(umap_2d), dtype=bool)

    kept_frac = keep_mask.mean()
    removed = (~keep_mask).sum()
    if removed > 0:
        print(f"[INFO] {model_tag}: removing {removed} outliers ({(1-kept_frac):.1%}) from plot")

    # Save FILTERED coords (for transparency)
    filt_coords_out = os.path.join(out_dir, f"umap_coords_{model_tag}_filtered.csv")
    pd.DataFrame({
        "sample_path": df_join.loc[keep_mask, "sample_path"].values,
        "umap_x": umap_2d[keep_mask, 0],
        "umap_y": umap_2d[keep_mask, 1],
    }).to_csv(filt_coords_out, index=False)

    # Use filtered arrays for plotting only
    U = umap_2d[keep_mask]
    DF = df_join.loc[keep_mask].reset_index(drop=True)

    for task in tasks:
        if task not in DF.columns:
            print(f"[WARN] {model_tag}: task '{task}' not in metadata; skipping.")
            continue

        labels = DF[task].astype("string").fillna("NA")
        kept = KEPT_BY_TASK.get(task, [])
        color_map = CMAP_BY_TASK.get(task, {})

        is_keep = labels.isin(kept).values

        fig, ax = plt.subplots(figsize=(7.8, 6.6))

        # gray background for non-kept classes (still filtered by outliers)
        m_bg = ~is_keep
        if m_bg.any():
            ax.scatter(
                U[m_bg, 0], U[m_bg, 1],
                s=6, alpha=0.25, linewidths=0, c="#cfcfcf"
            )

        # kept classes with fixed colors
        for lab in kept:
            sel = (labels.values == lab)
            if not sel.any():
                continue
            ax.scatter(
                U[sel, 0], U[sel, 1],
                s=6, alpha=0.8, linewidths=0,
                color=color_map[lab], label=str(lab)
            )

        ax.set_title(f"UMAP — {model_tag} [{task}]", fontsize=20, weight="bold")
        ax.set_xlabel("UMAP-1", fontsize=20, weight="bold")
        ax.set_ylabel("UMAP-2", fontsize=20, weight="bold")

        # Increase tick label sizes
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)

        plt.setp(ax.get_xticklabels(), fontweight='bold')
        plt.setp(ax.get_yticklabels(), fontweight='bold')

        if 0 < len(kept) <= 25:
            ax.legend(
                markerscale=3,
                bbox_to_anchor=(1.02, 1),
                loc="upper left",
                fontsize=8,
                frameon=False,
                borderaxespad=0.0
            )

        plt.tight_layout()
        out_path = os.path.join(out_dir, f"umap_{model_tag}_{task}.png")
        plt.savefig(out_path, dpi=250, bbox_inches="tight", pad_inches=0.0)
        plt.close()
        print(f"[OK] saved: {out_path}")

# -----------------------------
# 4) ARI experiment (per model & per task)
# -----------------------------
def compute_ari_per_task(
    emb: np.ndarray,
    df_join: pd.DataFrame,
    tasks,
    kept_by_task,
    max_samples=ARI_MAX_SAMPLES_PER_TASK,
    seed=RANDOM_SEED,
) -> dict:
    """
    For a given model embedding matrix (emb) and joined metadata (df_join),
    cluster in embedding space (KMeans) and compute ARI vs. ground-truth labels
    for each task. Only uses 'kept' classes (same as UMAPs/pies).
    """
    rng = np.random.default_rng(seed)
    ari_scores = {}

    for task in tasks:
        if task not in df_join.columns:
            continue

        labels_full = df_join[task].astype("string").fillna("NA")
        kept_classes = kept_by_task.get(task, [])
        if len(kept_classes) < 2:
            # need at least 2 classes for ARI
            continue

        mask = labels_full.isin(kept_classes).values
        n_kept = mask.sum()
        if n_kept < 100:   # too few to be meaningful
            print(f"[WARN] ARI: {task} has only {n_kept} samples after filtering; skipping.")
            continue

        X = emb[mask]
        y = labels_full[mask].to_numpy()

        # Optional subsampling for speed
        if n_kept > max_samples:
            sub_idx = rng.choice(n_kept, size=max_samples, replace=False)
            X = X[sub_idx]
            y = y[sub_idx]
            n_kept = len(y)

        n_clusters = len(kept_classes)
        # If degenerate, skip
        if n_clusters < 2 or n_kept <= n_clusters:
            print(f"[WARN] ARI: {task} has n_clusters={n_clusters}, n_samples={n_kept}; skipping.")
            continue

        km = KMeans(
            n_clusters=n_clusters,
            n_init=ARI_N_INIT,
            random_state=seed,
            verbose=0,
        )
        y_pred = km.fit_predict(X)

        ari = adjusted_rand_score(y, y_pred)
        ari_scores[task] = ari
        print(f"[INFO] ARI {task}: {ari:.3f} (n={n_kept}, K={n_clusters})")

    return ari_scores

# -----------------------------
# Main: loop models (UMAP once per model) + ARI + global pies
# -----------------------------
ARI_RESULTS = []   # rows: {"model": tag, task1: ari, ...}

for m in MODELS:
    tag = m["tag"]
    emb_dir = m["emb_dir"]
    feats_file = m["feats_file"]

    feats_path = os.path.join(emb_dir, feats_file)
    index_path = os.path.join(emb_dir, "index.csv")
    if not (os.path.exists(feats_path) and os.path.exists(index_path)):
        print(
            f"[WARN] Missing artifacts for {tag} at {emb_dir} "
            f"(need {feats_file} + index.csv)"
        )
        continue

    emb = np.load(feats_path)
    index = pd.read_csv(index_path)  # must have 'sample_path'

    # Join & align
    df_join = meta.merge(index, on="sample_path", how="inner").reset_index(drop=True)
    if "index" in df_join.columns:
        df_join = df_join.drop(columns=["index"])

    # Align lengths robustly (index.csv order is assumed to match emb row order)
    if emb.shape[0] != len(df_join):
        if emb.shape[0] > len(df_join):
            print(
                f"[WARN] {tag}: more embeddings ({emb.shape[0]}) than metadata rows ({len(df_join)}); trimming embeddings."
            )
            emb = emb[:len(df_join)]
        else:
            print(
                f"[WARN] {tag}: fewer embeddings ({emb.shape[0]}) than metadata rows ({len(df_join)}); truncating metadata."
            )
            df_join = df_join.iloc[:emb.shape[0]].reset_index(drop=True)

    # ----- ARI experiment (unsupervised clustering vs labels) -----
    ari_scores = compute_ari_per_task(
        emb=emb,
        df_join=df_join,
        tasks=TASKS,
        kept_by_task=KEPT_BY_TASK,
        max_samples=ARI_MAX_SAMPLES_PER_TASK,
        seed=RANDOM_SEED,
    )
    row = {"model": tag}
    row.update(ari_scores)
    ARI_RESULTS.append(row)

    # Optional FAST subsampling for UMAP fit (fit on subset, transform all)
    SUB_FIT = None
    if FAST_MODE and emb.shape[0] > 120_000:
        rng = np.random.default_rng(RANDOM_SEED)
        sub_idx = rng.choice(emb.shape[0], size=120_000, replace=False)
        SUB_FIT = sub_idx

    reducer = umap.UMAP(
        n_neighbors=N_NEIGHBORS, min_dist=MIN_DIST,
        random_state=RANDOM_SEED, metric=METRIC
    )
    if SUB_FIT is None:
        umap_2d = reducer.fit_transform(emb)
    else:
        reducer.fit(emb[SUB_FIT])
        umap_2d = reducer.transform(emb)

    print(f"[INFO] UMAP {tag}: n={emb.shape[0]} → coords computed once")

    # Plot SAME coords, recolor by each task (colors fixed globally)
    plot_same_coords_color_by_tasks(
        umap_2d=umap_2d,
        df_join=df_join,
        model_tag=tag,
        out_dir=OUT_DIR,
        tasks=TASKS,
        min_class_count=MIN_CLASS_COUNT,
        outlier_remove=OUTLIER_REMOVE
    )

# -----------------------------
# ARI summary table + publication-ready heatmap
# -----------------------------
if ARI_RESULTS:
    ari_df = pd.DataFrame(ARI_RESULTS).set_index("model")

    # Save numeric values
    ari_csv_path = os.path.join(OUT_DIR, "ari_scores_per_model_per_task.csv")
    ari_df.to_csv(ari_csv_path)
    print(f"[OK] ARI table saved to: {ari_csv_path}")

    # Reorder columns to follow TASKS (only keep those that exist)
    cols = [t for t in TASKS if t in ari_df.columns]
    ari_df = ari_df[cols]

    # Publication-ready heatmap
    plt.figure(
        figsize=(
            max(6, 1.4 * len(cols) + 2),
            max(4, 0.7 * len(ari_df.index) + 2),
        )
    )

    hm = sns.heatmap(
        ari_df,
        annot=True,
        fmt=".3f",
        vmin=0.0,
        vmax=1.0,
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": "Adjusted Rand Index"},
        square=False,
    )

    # Styling
    plt.title("Unsupervised Clustering Agreement (ARI)", fontsize=18, weight="bold")
    plt.xticks(rotation=45, ha="right", fontsize=12, weight="bold")
    plt.yticks(rotation=0, fontsize=12, weight="bold")

    cbar = hm.collections[0].colorbar
    cbar.ax.tick_params(labelsize=11)
    cbar.ax.set_ylabel("Adjusted Rand Index", fontsize=12, weight="bold")

    plt.tight_layout()
    ari_fig_path = os.path.join(OUT_DIR, "ari_heatmap_models_vs_tasks.png")
    plt.savefig(ari_fig_path, dpi=350, bbox_inches="tight", pad_inches=0.08)
    plt.close()
    print(f"[OK] ARI heatmap saved to: {ari_fig_path}")
else:
    print("[WARN] No ARI results computed; skipping ARI heatmap.")

# -------- After UMAPs & ARI: ONE pie per task (colors match UMAPs) --------
for task in TASKS:
    save_global_pie_for_task(meta, task, KEPT_BY_TASK, CMAP_BY_TASK, OUT_DIR)


In [ ]:
# =====================================================
# Grouped ARI barplot (grouped by TASK, single figure)
# x-axis = tasks, bars = models
# with fuzzy model selection + custom display names
# =====================================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pandas as pd

# ---- OPTIONAL: choose which models to plot (order matters) ----
# You can use either exact names or distinctive substrings.
# Examples:
PLOT_MODELS = [
     "msi_multitask_mae_untrained",
     "msi_multitask_dinov2_untrained",
     "msi_multitask_mae",
     "msi_multitask_dinov2"
]

# ---- OPTIONAL: pretty names for legend (key = ari_df index) ----
# If you use substrings in PLOT_MODELS, the *matched* full names
# are the keys here.
MODEL_LABEL_MAP = {
    "msi_multitask_mae_untrained": "MAE-ViT-B/16 – Pretrained",
    "msi_multitask_dinov2_untrained": "DINOv2-ViT-B/14 – Pretrained",
    "msi_multitask_mae": "MAE-ViT-B/16 – Fine-tuned",
    "msi_multitask_dinov2": "DINOv2-ViT-B/14 – Fine-tuned"
}

# If ari_df already exists, comment these two lines:
ari_csv_path = os.path.join(OUT_DIR, "ari_scores_per_model_per_task.csv")
ari_df = pd.read_csv(ari_csv_path, index_col="model")

available_models = list(ari_df.index)
print("[INFO] Available models in ari_df:")
for m in available_models:
    print("  -", m)

# ---- Helper: fuzzy model selection ----
def select_models(ari_df, desired):
    if not desired:
        return list(ari_df.index)

    desired_clean = [d.strip() for d in desired if d.strip()]
    matched = []

    for d in desired_clean:
        # 1) exact match
        if d in ari_df.index:
            matched.append(d)
            continue
        # 2) substring match
        candidates = [m for m in ari_df.index if d in m]
        matched.extend(candidates)

    # preserve order & deduplicate
    seen = set()
    ordered_unique = []
    for m in matched:
        if m not in seen:
            seen.add(m)
            ordered_unique.append(m)

    if not ordered_unique:
        print("[WARN] PLOT_MODELS did not match any models; using ALL models instead.")
        return list(ari_df.index)

    print("[INFO] Using models:")
    for m in ordered_unique:
        print("  -", m)
    return ordered_unique

# ---- Apply selection ----
models_order = select_models(ari_df, PLOT_MODELS)
ari_df_plot = ari_df.loc[models_order]

# ---- Keep only tasks present ----
task_cols = [t for t in TASKS if t in ari_df_plot.columns]
if not task_cols:
    raise SystemExit("[ERR] No ARI task columns found for grouped-by-task barplot.")

models = ari_df_plot.index.tolist()
n_models = len(models)
n_tasks = len(task_cols)

sns.set_style("whitegrid")
plt.figure(figsize=(max(10, 2 * n_tasks), 6))

# X positions for TASK groups
x = np.arange(n_tasks)
width = 0.8 / n_models   # bar width scaled by number of models

# Color palette per model
palette = sns.color_palette("tab10", n_colors=n_models)

handles_for_legend = []
labels_for_legend = []

# ---- Plot grouped bars: for each model, bars over tasks ----
for i, model in enumerate(models):
    vals = ari_df_plot.loc[model, task_cols].values
    pretty_label = MODEL_LABEL_MAP.get(model, model)

    bars = plt.bar(
        x + i * width,
        vals,
        width=width,
        label=pretty_label,
        color=palette[i],
        edgecolor="white"
    )
    # Keep one handle for legend
    handles_for_legend.append(bars[0])
    labels_for_legend.append(pretty_label)

    # Value labels
    for xt, yt in zip(x + i * width, vals):
        plt.text(
            xt,
            yt + 0.015,
            f"{yt:.3f}",
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="bold"
        )

# Center ticks at group centers
plt.xticks(
    x + (width * (n_models - 1) / 2),
    task_cols,
    rotation=45,
    ha="right",
    fontsize=11,
    fontweight="bold"
)
plt.yticks(fontsize=11, fontweight="bold")
plt.ylim(0, 0.25)

plt.ylabel("Adjusted Rand Index (ARI)", fontsize=13, fontweight="bold")
plt.xlabel("Task", fontsize=13, fontweight="bold")
plt.title("Unsupervised Clustering Agreement per Task", fontsize=15, fontweight="bold")

plt.legend(
    handles=handles_for_legend,
    labels=labels_for_legend,
    title="Model",
    fontsize=9,
    title_fontsize=10,
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
out_bar = os.path.join(OUT_DIR, "ari_grouped_barplot_by_task.png")
plt.savefig(out_bar, dpi=350, bbox_inches="tight", pad_inches=0.1)
plt.close()
print(f"[OK] grouped-by-task ARI barplot saved to: {out_bar}")